# 01 — Data Quality, Target Definition and Leakage Audit

**Question for this notebook:** is the extracted data fit for building a default model, and which columns may legitimately be used?

Scope: Lending Club 36-month loans issued 2012–2015. Bad = Charged Off / Default / Late (31-120 days); Good = Fully Paid; other statuses excluded (outcome not mature).

In [15]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
from statsmodels.stats.proportion import proportion_confint

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config
from src.data_prep import classify_columns, download, extract, quality_connection

download()
extract()
con = quality_connection()
pd.set_option("display.max_rows", 200)

## 1. Leakage and column audit
Every column is assigned to one group in `src/config.py`. Anything not listed becomes a *candidate feature* and must be checked by hand.

In [16]:
schema = con.sql("DESCRIBE loans").df()[["column_name", "column_type"]]
audit = (
    classify_columns(schema["column_name"].tolist())
    .merge(schema, left_on="column", right_on="column_name")
    .drop(columns="column_name")
)
listed = {c for cols in config.COLUMN_GROUPS.values() for c in cols}
print("Listed in config but absent from data:", sorted(listed - set(audit["column"])))
audit["group"].value_counts()

Listed in config but absent from data: []


group
candidate_feature             62
post_origination_leakage      39
joint_application             16
unavailable_in_development    14
administrative                 8
id_or_free_text                6
derived                        5
lc_risk_assessment             4
fairness_proxy                 2
Name: count, dtype: int64

In [21]:
audit[audit["group"] == "candidate_feature"].reset_index(drop=True)

,column,group,column_type
0,loan_amnt,candidate_feature,DOUBLE
1,emp_length,candidate_feature,VARCHAR
2,home_ownership,candidate_feature,VARCHAR
3,annual_inc,candidate_feature,DOUBLE
4,verification_status,candidate_feature,VARCHAR
5,purpose,candidate_feature,VARCHAR
6,dti,candidate_feature,DOUBLE
7,delinq_2yrs,candidate_feature,DOUBLE
8,earliest_cr_line,candidate_feature,VARCHAR
9,fico_range_low,candidate_feature,DOUBLE


## 2. Outcome maturity: loan status by issue year

In [20]:
con.sql("SELECT * FROM dq_status_by_year").df()

,issue_year,loan_status,n,pct_of_year
0,2012,Fully Paid,37567,86.42
1,2012,Charged Off,5903,13.58
2,2013,Fully Paid,88044,87.67
3,2013,Charged Off,12378,12.33
4,2014,Fully Paid,140255,86.27
5,2014,Charged Off,22315,13.73
6,2015,Fully Paid,240894,85.07
7,2015,Charged Off,42132,14.88
8,2015,Current,72,0.03
9,2015,Late (31-120 days),61,0.02


## 3. Vintage table with 95% Wilson confidence intervals

In [18]:
def add_wilson_ci(df: pd.DataFrame) -> pd.DataFrame:
    resolved = df["n_good"] + df["n_bad"]
    low, high = proportion_confint(df["n_bad"], resolved, method="wilson")
    return df.assign(
        bad_rate_pct=(100 * df["n_bad"] / resolved).round(2),
        ci_low_pct=(100 * low).round(2),
        ci_high_pct=(100 * high).round(2),
    )

vintage = add_wilson_ci(con.sql("SELECT * FROM dq_vintage").df())
vintage

,sample,issue_quarter,n_total,n_excluded,n_good,n_bad,pct_excluded,bad_rate_pct,ci_low_pct,ci_high_pct
0,pre_development,2012-Q1,6498,0,5640,858,0.00,13.20,12.40,14.05
1,pre_development,2012-Q2,8469,0,7264,1205,0.00,14.23,13.50,14.99
2,pre_development,2012-Q3,13322,0,11431,1891,0.00,14.19,13.61,14.80
3,development,2012-Q4,15181,0,13232,1949,0.00,12.84,12.32,13.38
4,development,2013-Q1,17988,0,15724,2264,0.00,12.59,12.11,13.08
5,development,2013-Q2,23136,0,20166,2970,0.00,12.84,12.41,13.27
6,development,2013-Q3,27318,0,23982,3336,0.00,12.21,11.83,12.61
7,development,2013-Q4,31980,0,28172,3808,0.00,11.91,11.56,12.27
8,oot,2014-Q1,34074,0,29690,4384,0.00,12.87,12.51,13.23
9,oot,2014-Q2,37881,0,32750,5131,0.00,13.55,13.20,13.89


In [19]:
by_sample = (
    vintage.groupby("sample", sort=False)[["n_total", "n_excluded", "n_good", "n_bad"]].sum()
    .pipe(add_wilson_ci)
    .assign(pct_excluded=lambda d: (100 * d["n_excluded"] / d["n_total"]).round(2))
)
by_sample

,n_total,n_excluded,n_good,n_bad,bad_rate_pct,ci_low_pct,ci_high_pct,pct_excluded
sample,,,,,,,,
pre_development,28289,0,24335,3954,13.98,13.58,14.39,0.00
development,115603,0,101276,14327,12.39,12.20,12.58,0.00
oot,162570,0,140255,22315,13.73,13.56,13.89,0.00
monitoring,283173,86,240894,42193,14.90,14.77,15.04,0.03


## 4. Application type

In [22]:
con.sql("SELECT * FROM dq_application_type").df()

,issue_year,application_type,n
0,2012,Individual,43470
1,2013,Individual,100422
2,2014,Individual,162570
3,2015,Individual,282934
4,2015,Joint App,239


## 5. Missingness of candidate features by sample (%)
Features that are mostly missing in development, or whose missingness shifts between samples, are a data-quality and stability risk.

In [23]:
candidates = audit.loc[audit["group"] == "candidate_feature", "column"].tolist()
null_shares = ", ".join(f'AVG(CASE WHEN "{c}" IS NULL THEN 1.0 ELSE 0 END) AS "{c}"' for c in candidates)
missing = (
    con.sql(f"SELECT sample, {null_shares} FROM loans WHERE bad_flag IS NOT NULL GROUP BY sample").df()
    .set_index("sample").T.mul(100).round(1)
    [["development", "oot", "monitoring"]]
)
missing["max_shift_pp"] = missing.max(axis=1) - missing.min(axis=1)
missing.sort_values(["development", "max_shift_pp"], ascending=False)

sample,development,oot,monitoring,max_shift_pp
mths_since_last_record,88.7,81.1,81.3,7.6
mths_since_last_major_derog,79.1,70.7,69.8,9.3
mths_since_recent_bc_dlq,76.7,73.1,73.8,3.6
mths_since_recent_revol_delinq,68.8,63.5,63.4,5.4
mths_since_last_delinq,55.2,48.9,48.0,7.2
mths_since_recent_inq,11.4,9.5,11.0,1.9
emp_length,4.9,5.9,6.7,1.8
mo_sin_old_il_acct,4.3,3.6,3.5,0.8
bc_util,0.9,1.2,1.1,0.3
bc_open_to_buy,0.8,1.1,1.0,0.3


In [24]:
con.sql("""
    SELECT
        issue_quarter,
        ROUND(100 * AVG(CASE WHEN tot_cur_bal IS NULL THEN 1.0 ELSE 0 END), 1) AS pct_missing_tot_cur_bal,
        ROUND(100 * AVG(CASE WHEN num_sats IS NULL THEN 1.0 ELSE 0 END), 1) AS pct_missing_num_sats,
        ROUND(100 * AVG(CASE WHEN mort_acc IS NULL THEN 1.0 ELSE 0 END), 1) AS pct_missing_mort_acc,
        ROUND(100 * AVG(CASE WHEN open_acc_6m IS NULL THEN 1.0 ELSE 0 END), 1) AS pct_missing_open_acc_6m
    FROM loans
    WHERE bad_flag IS NOT NULL
    GROUP BY issue_quarter
    ORDER BY issue_quarter
""").df()

,issue_quarter,pct_missing_tot_cur_bal,pct_missing_num_sats,pct_missing_mort_acc,pct_missing_open_acc_6m
0,2012-Q1,100.0,100.0,92.0,100.0
1,2012-Q2,100.0,76.3,0.5,100.0
2,2012-Q3,56.9,0.1,0.0,100.0
3,2012-Q4,0.0,0.0,0.0,100.0
4,2013-Q1,0.0,0.0,0.0,100.0
5,2013-Q2,0.0,0.0,0.0,100.0
6,2013-Q3,0.0,0.0,0.0,100.0
7,2013-Q4,0.0,0.0,0.0,100.0
8,2014-Q1,0.0,0.0,0.0,100.0
9,2014-Q2,0.0,0.0,0.0,100.0


## 6. Ranges and duplicates

In [25]:
print("Duplicate ids:", con.sql("SELECT COUNT(*) FROM dq_duplicate_ids").fetchall()[0][0])
con.sql("SELECT * FROM dq_ranges").df().T

Duplicate ids: 0


,0
min_annual_inc,0.0
max_annual_inc,9000000.0
n_income_nonpositive,2.0
min_dti,0.0
max_dti,999.0
n_dti_out_of_range,5.0
max_revol_util,892.3
n_revol_util_over_100,2208.0
min_fico,660.0
max_fico,850.0


## 7. Findings

1. **Column audit.** Of 151 source columns, 62 are candidate application-time features. 39 post-origination fields (payments, recoveries, last FICO, hardship, settlement) are excluded as outcome leakage; LC's `grade`/`sub_grade`/`int_rate`/`installment` are kept only as a benchmark; `zip_code`/`addr_state` are excluded as potential proxies for protected characteristics.
2. **Feature rollout changes the development window.** About 25 bureau fields (e.g. `tot_cur_bal`, `num_sats`) are 100% missing in 2012-Q1/Q2 and partly in Q3. Development therefore starts at **2012-Q4**; 2012-Q1..Q3 is kept as `pre_development` and not used. 14 further fields (e.g. `open_acc_6m`, `il_util`) only appear in 2015-Q4 and are excluded entirely — a model cannot use inputs that did not exist when it was built.
3. **Outcomes are mature.** Only 86 loans (0.03%, all 2015) have an unresolved status and are excluded. The `Default` status never occurs; "bad" is effectively *Charged Off* (plus 61 loans 31–120 days late).
4. **The bad rate rises over time, beyond sampling noise.** Development 12.39% (95% CI 12.20–12.58%), OOT 13.73% (13.56–13.89%), monitoring 14.90% (14.77–15.04%). The intervals do not overlap, so this is a genuine shift. A model calibrated on development data is therefore expected to **underestimate** PD later; the calibration tests and monitoring must detect this.
5. **Structural missingness is information, not error.** `mths_since_*` fields are missing when the event never happened (e.g. 55% of development borrowers have no recorded delinquency). These get their own bin rather than imputation. Their missing share falls from development to OOT (e.g. `mths_since_last_delinq` 55% → 49%): more borrowers with prior delinquencies — an early population-drift signal.
6. **Data-quality watch item.** `num_tl_120dpd_2m` missingness rises from 0.2% (development) to 5.0% (monitoring).
7. **Invalid values.** 2 loans with income ≤ 0 and 5 with DTI outside 0–100 (max 999) are set to missing. 2,208 loans with `revol_util` > 100% are plausible (over-limit) and kept; binning caps their effect.
8. **Selection bias.** Minimum FICO is 660: only loans Lending Club *accepted* are observed. The model learns risk within an already-screened population (no reject inference) — a stated limitation.
9. **Scope filters.** 239 joint applications (2015 only) excluded; no duplicate loan IDs.

**Modelling samples** (resolved, individual loans): development 115,603 (14,327 bad), OOT 162,570 (22,315 bad), monitoring 282,848 (42,150 bad).